# MLP Architecture and Parameter Counting

Companion notebook for Section 2 of the MLP lecture notes.

Objectives:

- represent an MLP architecture as a sequence of layer sizes;
- compute the shapes of weight matrices and bias vectors;
- count learnable parameters layer by layer;
- compare width and depth;
- verify parameter counts with scikit-learn and PyTorch-style formulas.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True


## 1. Dense layers as matrix transformations

For a dense layer with `n_in` inputs and `n_out` neurons:

- `W` has shape `(n_out, n_in)` using the lecture/PyTorch convention;
- `b` has shape `(n_out,)`;
- parameter count is `n_in * n_out + n_out`.


In [ ]:
def layer_parameter_count(n_in, n_out):
    weights = n_in * n_out
    biases = n_out
    return weights + biases

print('3 inputs -> 4 neurons:', layer_parameter_count(3, 4), 'parameters')
print('4 inputs -> 2 neurons:', layer_parameter_count(4, 2), 'parameters')


## 2. Architecture summary helper

We encode an architecture as `[input_dim, hidden_1, hidden_2, ..., output_dim]`. The input layer has no learnable parameters.


In [ ]:
def architecture_summary(layer_sizes):
    rows = []
    for layer_index, (n_in, n_out) in enumerate(zip(layer_sizes[:-1], layer_sizes[1:]), start=1):
        rows.append({
            'layer': layer_index,
            'n_in': n_in,
            'n_out': n_out,
            'W_shape_lecture_pytorch': (n_out, n_in),
            'b_shape': (n_out,),
            'weights': n_in * n_out,
            'biases': n_out,
            'parameters': n_in * n_out + n_out,
        })
    return pd.DataFrame(rows)

architecture = [3, 4, 2]
summary = architecture_summary(architecture)
summary


In [ ]:
print('Total parameters:', summary['parameters'].sum())


## 3. Worked example from the lecture notes

A network with 3 inputs, one hidden layer with 4 neurons, and an output layer with 2 neurons has:

- first layer: `4 * 3 + 4 = 16`;
- output layer: `2 * 4 + 2 = 10`;
- total: `26`.


In [ ]:
lecture_example = architecture_summary([3, 4, 2])
lecture_example


## 4. Entry Ticket 2 architecture

Input dimension `d = 4`, hidden layers with 8 and 4 neurons, output layer with 3 neurons.


In [ ]:
entry_ticket_arch = [4, 8, 4, 3]
entry_ticket_summary = architecture_summary(entry_ticket_arch)
entry_ticket_summary


In [ ]:
print('Total parameters:', entry_ticket_summary['parameters'].sum())

for _, row in entry_ticket_summary.iterrows():
    print(f"W^{int(row['layer'])} shape = {row['W_shape_lecture_pytorch']}, b^{int(row['layer'])} shape = {row['b_shape']}")


## 5. Doubling hidden-layer widths

The parameter count usually grows more than linearly because adjacent dense layers multiply widths.


In [ ]:
doubled_hidden_arch = [4, 16, 8, 3]
comparison = pd.DataFrame([
    {'architecture': str(entry_ticket_arch), 'total_parameters': architecture_summary(entry_ticket_arch)['parameters'].sum()},
    {'architecture': str(doubled_hidden_arch), 'total_parameters': architecture_summary(doubled_hidden_arch)['parameters'].sum()},
])
comparison['relative_to_original'] = comparison['total_parameters'] / comparison.loc[0, 'total_parameters']
comparison


## 6. Width versus depth

Two networks can have similar or very different parameter counts depending on how width and depth are allocated.


In [ ]:
architectures = {
    'shallow narrow': [20, 8, 2],
    'shallow wide': [20, 64, 2],
    'deep narrow': [20, 16, 16, 16, 2],
    'deep wider': [20, 64, 32, 16, 2],
}

rows = []
for name, sizes in architectures.items():
    rows.append({
        'name': name,
        'layer_sizes': sizes,
        'hidden_layers': len(sizes) - 2,
        'max_width': max(sizes[1:-1]),
        'total_parameters': architecture_summary(sizes)['parameters'].sum(),
    })

arch_df = pd.DataFrame(rows)
arch_df


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(arch_df['name'], arch_df['total_parameters'])
ax.set_ylabel('Learnable parameters')
ax.set_title('Parameter count depends on width and depth')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


## 7. Verify with scikit-learn

Scikit-learn stores dense-layer weights in `coefs_`, with shape `(n_in, n_out)`. This is the transpose of the lecture/PyTorch convention, but the parameter count is identical.


In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier

X_dummy = np.random.default_rng(42).normal(size=(30, 4))
y_dummy = np.random.default_rng(43).integers(0, 3, size=30)

clf = MLPClassifier(hidden_layer_sizes=(8, 4), max_iter=500, random_state=42)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    clf.fit(X_dummy, y_dummy)

sklearn_rows = []
for i, (coef, intercept) in enumerate(zip(clf.coefs_, clf.intercepts_), start=1):
    sklearn_rows.append({
        'layer': i,
        'coef_shape_sklearn': coef.shape,
        'intercept_shape': intercept.shape,
        'parameters': coef.size + intercept.size,
    })

pd.DataFrame(sklearn_rows)


In [ ]:
manual_total = architecture_summary([4, 8, 4, 3])['parameters'].sum()
sklearn_total = sum(coef.size + intercept.size for coef, intercept in zip(clf.coefs_, clf.intercepts_))
print('Manual total:', manual_total)
print('scikit-learn total:', sklearn_total)


## 8. PyTorch-style module count, without requiring PyTorch

A PyTorch `nn.Linear(n_in, n_out)` stores a weight tensor with shape `(n_out, n_in)` and a bias tensor with shape `(n_out,)`, matching the convention used in the lecture notes.


In [ ]:
def pytorch_linear_shapes(layer_sizes):
    rows = []
    for i, (n_in, n_out) in enumerate(zip(layer_sizes[:-1], layer_sizes[1:]), start=1):
        rows.append({
            'module': f'Linear_{i}',
            'weight_shape': (n_out, n_in),
            'bias_shape': (n_out,),
            'parameters': n_out * n_in + n_out,
        })
    return pd.DataFrame(rows)

pytorch_linear_shapes([4, 8, 4, 3])


## Exercises

1. Compute the parameter count for `[10, 128, 64, 3]`. Which layer has most parameters?
2. Compare `[100, 10, 2]` and `[100, 10, 10, 2]`. How many parameters are added by the extra hidden layer?
3. For a binary classifier, what changes if the output layer uses 1 neuron instead of 2?


## Takeaways

- Dense layers learn one weight per input-output neuron pair plus one bias per output neuron.
- The input layer has no learnable parameters.
- Width and depth affect both expressive capacity and overfitting risk through the number of learnable parameters.
